# Production-Grade RAG for Finance

- Fully local pipeline: Ollama runs the model and the embeddings, Qdrant holds the vectors, RAGWire wires it together
- Real corpus: the 2024 10-K filings of Amazon, Alphabet, and Meta
- Every finalized piece of code also lives in `scripts/`, so the notebook, the CLI, and the Chainlit app share one implementation
- LangSmith tracing is on: every model call in this session lands in the `finance-rag-bootcamp` project

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## 1. Text extraction: PDF to markdown

- A model cannot read PDF bytes. Step one is always text
- MarkItDown converts PDF, DOCX, XLSX, PPTX to markdown-ish text
- Finance is the hard case: a 10-K is hundreds of pages of prose wrapped around numbers

In [ ]:
from ragwire.loaders.markitdown_loader import MarkItDownLoader

loader = MarkItDownLoader()
doc = loader.load("data/amazon 10-k 2024.pdf")
len(doc["text_content"])

In [ ]:
doc

In [ ]:
text = doc["text_content"]
print(text[:700])

- That code is now a function in `scripts/extraction.py`, and from here on we import it

In [ ]:
from scripts.extraction import pdf_to_markdown

## 2. Chunking

- The whole filing does not fit in one prompt, so we cut it into chunks
- Where you cut decides what retrieval can ever find
- This pipeline uses the page strategy, new in RAGWire 1.6: exactly one chunk per PDF page, never sub-split, merged, or overlapped
- MarkItDown flattens a PDF into one string, so the page loader reads it again with pypdf, page by page, and a retrieved chunk then maps onto a page a reader can open

In [ ]:
from ragwire.loaders.page_loader import PageLoader
from ragwire.processing.splitter import PageSplitter

page_loader = PageLoader()
doc = page_loader.load("data/amazon 10-k 2024.pdf")

splitter = PageSplitter()
pages = splitter.split(doc["text_content"], pages=doc["pages"])
len(pages)

In [ ]:
# pages

In [ ]:
one = pages[41]
one["page_number"], one["page_total"]

In [ ]:
print(one["text"][:1600])
# print(pages[17]['text'])

- Same code, as a function, in `scripts/chunking.py`

In [ ]:
from scripts.chunking import split_pages

## 3. Embeddings

- A chunk is stored as a vector: a list of numbers that captures meaning
- nomic-embed-text runs on local Ollama and produces 768 numbers per text
- Similar meaning gives similar vectors, and that is what makes search by meaning possible

In [ ]:
from ragwire.embeddings.factory import get_embedding

embedder = get_embedding({"provider": "ollama", "model": "nomic-embed-text"})
vector = embedder.embed_query("What was Amazon's revenue in 2024?")
len(vector)

In [ ]:
# vector

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

v1 = embedder.embed_query("revenue increased during the year")
v2 = embedder.embed_query("net sales grew over the period")
v3 = embedder.embed_query("the cafeteria serves lunch at noon")

cosine_similarity([v1], [v2]), cosine_similarity([v1], [v3])

- Two sentences that mean the same thing score far higher than an unrelated one
- `scripts/embeddings.py` returns this same embedder

## 4. RAGWire: the pieces become one pipeline

- Everything so far was a RAGWire building block used by hand
- One config file, `config/finance_rag.yaml`, assembles them: loader, splitter, embeddings, vector store, LLM
- The commented lines in that file show the options you are not using

In [ ]:
from ragwire import RAGWire

rag = RAGWire("config/finance_rag.yaml")

- During ingestion the LLM reads each filing's cover page and extracts metadata
- The instructions live in `config/metadata.yaml`, so the schema is yours to change
- Run the extractor by hand first, on the text from section 1, so ingestion is not a black box

In [ ]:
len(text)

In [ ]:
rag.extract_metadata(text[:3000])

- Ingest one filing first and look at what it produced

In [ ]:
rag.ingest_documents(["data/amazon 10-k 2024.pdf"])

In [ ]:
rag.get_stats()

## 5. Ingesting the rest, and living with change

- Point ingestion at the whole directory. The filing we already ingested is hashed, recognized, and skipped

In [ ]:
rag.ingest_directory("data")

In [ ]:
rag.get_field_values(["company_name", "doc_type", "fiscal_year"])

- Filings get amended. Remove a document, or force one through again

In [ ]:
rag.delete_document("data/amazon 10-k 2024.pdf")

In [ ]:
rag.reingest_documents(["data/amazon 10-k 2024.pdf"])

## 6. Hybrid retrieval and metadata filters

- Every chunk carries two vectors: dense for meaning, sparse (BM25 style) for exact words
- Finance queries lean on exact tokens: AWS, 10-K, segment names, dollar figures
- `auto_filter` is off in this config, so `retrieve()` searches the whole collection unless you pass filters yourself

In [ ]:
docs = rag.retrieve("How did AWS perform in 2024?")

for d in docs:
    print(d.metadata["file_name"], "| page", d.metadata["page_number"])

In [ ]:
print(docs[0].page_content[:600])

In [ ]:
docs = rag.retrieve("How did apple perform in 2024?")
docs

- You can always pass filters yourself, built from the metadata that ingestion stored

In [ ]:
docs = rag.retrieve("How did advertising revenue change in 2024 of Google?",
                    filters={"company_name": "alphabet inc."})

for d in docs:
    print(d.metadata["file_name"], "| page", d.metadata["page_number"])

- `extract_filters` shows what the LLM would pull from a question. With `auto_filter: true` the pipeline would apply this by itself; section 9 hands the decision to an agent instead

In [ ]:
rag.extract_filters("How did Meta's advertising revenue change in 2024?")

In [ ]:
docs = rag.retrieve("What did Facebook say about revenue growth in 2024?", filters={'company_name': 'meta', 'fiscal_year': 2024})

for d in docs:
    print(d.metadata["file_name"], "| page", d.metadata["page_number"])

## 7. Grounded answers with citations

- `query()` retrieves, numbers the sources, and instructs the model to answer from them alone
- Every claim must carry a citation. When the answer is a number, that is not optional

In [ ]:
a = rag.query("What was Amazon's total net sales in 2024?")
print(a.formatted())

In [ ]:
a.citations

## 8. The guardrail: refusing what the corpus does not contain

- Apple is not in this collection
- With no filters, retrieval still returns the closest chunks it has, from the wrong companies
- The chunks reach the model, and the grounding prompt is what refuses

In [ ]:
b = rag.query("What was Apple's total revenue in 2024?")
b.refused

In [ ]:
print(b.text)

## 9. Agentic RAG: the model decides the filters

- Sections 6 to 8 chose filters by hand. An agent can make that decision itself
- Two tools: `get_filter_context` reports the metadata fields and the values the collection stores, `search_documents` runs retrieval with the filters the agent chose and labels each result with its file and page
- `create_agent` runs the loop: inspect the filters, search, answer
- Memory is a SQLite checkpointer on disk, so follow-up questions see the earlier turns

In [ ]:
from typing import Optional

from langchain.tools import tool


@tool
def get_filter_context(query: str) -> str:
    """Get available metadata fields, stored values, and filter suggestions for a query.

    Call this before search_documents when the query involves specific metadata
    (company, year, document type, etc.). Use the returned context to decide
    what filters to pass to search_documents.
    """
    return rag.get_filter_context(query)


@tool
def search_documents(query: str, filters: Optional[dict] = None) -> str:
    """Search the document knowledge base for relevant information.

    Args:
        query: The search query
        filters: Optional metadata filters decided from get_filter_context.
                 Pass {} or omit to search without filtering.
    """
    results = rag.retrieve(query, top_k=5, filters=filters)
    if not results:
        return "No relevant documents found."

    chunks = []
    for doc in results:
        source = doc.metadata.get("file_name", "unknown")
        page = doc.metadata.get("page_number")
        if page is not None:
            source = source + ", page " + str(page)
        chunks.append("[" + source + "]\n" + doc.page_content)

    return "\n\n---\n\n".join(chunks)

- Same code in `scripts/tools.py`, so the Chainlit app can hand the same tools to its own agent

In [ ]:
from scripts.tools import get_filter_context, search_documents

In [ ]:
import sqlite3
from pathlib import Path

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from langgraph.checkpoint.sqlite import SqliteSaver

model = ChatOllama(model="qwen3.8:27B", base_url="http://localhost:11434")

Path("db").mkdir(exist_ok=True)
Path("db/agent_memory.db").unlink(missing_ok=True)   # start the demo on a clean thread
conn = sqlite3.connect("db/agent_memory.db", check_same_thread=False)
checkpointer = SqliteSaver(conn)

agent = create_agent(
    model=model,
    tools=[get_filter_context, search_documents],
    system_prompt=(
        "You are a helpful financial document assistant. "
        "For complex questions, break them down into simple sub-questions and answer each one before forming a final answer. "
        "Always use search_documents to retrieve information before answering. "
        "Never answer from general knowledge. "
        "Use get_filter_context before search_documents when the query involves specific metadata (company, year, document type, etc.). "
        "When a question asks about the whole collection, or for the highest, lowest, or a comparison across companies, first call get_filter_context to see which companies the collection holds, then search every relevant company before answering. "
        "If no relevant documents are found, say so. Do not guess or fabricate an answer. "
        "Always cite the source document in your answer."
    ),
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "class-demo"}}

- `agent.invoke` waits in silence and hands back the end state. `agent.stream` can surface every step as it happens instead
- `stream_agent` wraps it in a generator: it always yields the answer, and by default it also yields every tool call and result, so you watch the agent decide. `verbose=False` gives the answer alone
- The events are plain tuples, so `print(*event)` is enough to watch it work. Tool results are truncated, the stream is for display

In [ ]:
def stream_agent(agent, query, config, verbose=True):
    """Run one query and yield the agent's progress as it happens."""
    for event in agent.stream({"messages": [HumanMessage(query)]},
                              config=config, stream_mode="updates"):
        for update in event.values():
            for message in update.get("messages", []):
                if message.type == "ai" and message.tool_calls:
                    if verbose:
                        for call in message.tool_calls:
                            yield "tool_call", call["name"], call["args"]
                elif message.type == "tool":
                    if verbose:
                        yield "tool_result", message.name, message.content[:200]
                elif message.type == "ai" and message.content:
                    yield "answer", message.content

- Same code in `scripts/agent.py`, so the notebook and the Chainlit app run one implementation

In [ ]:
from scripts.agent import stream_agent

In [ ]:
for event in stream_agent(agent, "What was Google's advertising revenue in 2024?",
                          config):
    print(*event)

- The follow-up only makes sense if the agent remembers the last answer

In [ ]:
for event in stream_agent(agent, "How does that compare to Amazon's total revenue in 2024?",
                          config):
    print(*event)

- A third question spans the whole collection, so the prompt rule sends the agent through every company before it answers

In [ ]:
for event in stream_agent(agent, "Which company in the collection had the highest total revenue in 2024?",
                          config):
    print(*event)

- The checkpointer holds the whole thread. `get_state` reads it back without running the agent

In [ ]:
messages = agent.get_state(config).values["messages"]
[type(m).__name__ for m in messages]

In [ ]:
messages

## 10. From notebook to app

- `scripts/pipeline.py` holds the finalized pipeline, `get_rag()`; `scripts/tools.py` builds the tools on top of it
- `app.py` is a Chainlit UI over the agent you just built: the same tools, the same SQLite memory, and the same `stream_agent`, with the tool events drawn as steps in the chat
- Each chat session gets its own memory thread
- Run it from a terminal:

```
uv run chainlit run app.py
```

## 11. Inspecting the MCP server

- RAGWire ships an MCP server over stdio: `ragwire mcp serve`
- MCP Inspector is the standard debugging UI: it launches the server as a child process, lists its tools, and lets you call them by hand
- Needs Node.js; run it from this folder:

```
npx -y @modelcontextprotocol/inspector -e LANGSMITH_TRACING=false -- uv run ragwire mcp serve --config config/finance_rag.yaml
```

- The browser opens on the inspector UI: Connect, then List Tools
- `search_documents`, `get_filter_context` and `collection_stats` return quickly; `answer_question` runs the LLM, about ten seconds warm
- `LANGSMITH_TRACING=false` is not optional: the tracer shells out to git for run metadata, and inside a stdio MCP server that subprocess never returns

## 12. The collection as a tool: MCP

- RAGWire ships an MCP server: `search_documents`, `answer_question`, `get_filter_context`, and `collection_stats` become tools Claude can call
- Add it to Claude in `claude_desktop_config.json`:

```
{
  "mcpServers": {
    "finance-rag": {
      "command": "D:\\Corporate Training\\2026\\finance-rag-bootcamp\\.venv\\Scripts\\ragwire.exe",
      "args": ["mcp", "serve", "--config",
               "D:\\Corporate Training\\2026\\finance-rag-bootcamp\\config\\finance_rag.yaml"],
      "env": {"LANGSMITH_TRACING": "false"}
    }
  }
}
```